# BGE-M3 encoding sandbox

Local setup (no Colab, no pip-in-notebook):

1. From the repo root: `uv sync` — installs everything from `pyproject.toml` / `uv.lock`, including this notebook's `smve_lab` package in editable mode.
2. Select the **"Python (smve-lab)"** Jupyter kernel for this notebook (registered via `python -m ipykernel install --user --name smve-lab`).

This notebook is for interactive exploration only — playing with the model API on a couple of sentences, then a small end-to-end run on scifact. For the real batch job, use `scripts/encode_scifact.py` from the terminal (`uv run python scripts/encode_scifact.py`) so a multi-minute run doesn't die with the kernel.

In [2]:
import os, json
import numpy as np
from tqdm import tqdm

In [ ]:
from smve_lab.bge_m3 import load_model
from smve_lab.config import EncodeConfig, resolve_device

cfg = EncodeConfig()
print("device:", resolve_device(cfg.device))
model = load_model(cfg)

In [4]:
sentences_1 = ["Defination of BM25"]
sentences_2 = ["BGE M3 is an embedding model supporting dense retrieval, lexical matching and multi-vector interaction.",
               "BM25 is a bag-of-words retrieval function that ranks a set of documents based on the query terms appearing in each document"]

In [5]:
output_sent_1 = model.encode(sentences_1, return_dense=True, return_sparse=True, return_colbert_vecs=True)
output_sent_2 = model.encode(sentences_2, return_dense=True, return_sparse=True, return_colbert_vecs=True)

In [6]:
output_sent_1

{'dense_vecs': array([[-0.010414, -0.0449  , -0.02428 , ..., -0.008255,  0.015015,
          0.011086]], dtype=float16),
 'lexical_weights': [defaultdict(int,
              {'262': np.float16(0.05014),
               '5983': np.float16(0.1367),
               '2320': np.float16(0.04495),
               '111': np.float16(0.06323),
               '90017': np.float16(0.2515),
               '2588': np.float16(0.3335)})],
 'colbert_vecs': [array([[ 0.01720414,  0.03835842, -0.02307892, ...,  0.00153593,
           0.02997066, -0.05984047],
         [ 0.00999266,  0.03920962, -0.03854785, ...,  0.00605929,
           0.02721512, -0.06508467],
         [ 0.01781991,  0.03922448, -0.01708746, ...,  0.00809134,
           0.03998277, -0.05070229],
         ...,
         [ 0.05480742,  0.00756531,  0.0033456 , ..., -0.01653274,
           0.02403149,  0.00371166],
         [ 0.00940319,  0.05026954, -0.02382976, ...,  0.02581409,
           0.00792389, -0.03253576],
         [ 0.01806276,  0.01

In [7]:
type(output_sent_1)

dict

In [8]:
output_sent_1.keys()

dict_keys(['dense_vecs', 'lexical_weights', 'colbert_vecs'])

In [9]:
output_sent_1["lexical_weights"]

[defaultdict(int,
             {'262': np.float16(0.05014),
              '5983': np.float16(0.1367),
              '2320': np.float16(0.04495),
              '111': np.float16(0.06323),
              '90017': np.float16(0.2515),
              '2588': np.float16(0.3335)})]

In [10]:
len(output_sent_2['colbert_vecs'][0])

30

In [11]:
sentence_pairs = [[i,j] for i in sentences_1 for j in sentences_2]

similarity_scores = model.compute_score(sentence_pairs,
                          max_passage_length=128, # a smaller max length leads to a lower latency
                          weights_for_different_modes=[0.4, 0.2, 0.4]) # weights_for_different_modes(w) is used to do weighted sum: w[0]*dense_score + w[1]*sparse_score + w[2]*colbert_score

print(similarity_scores)

{'colbert': [0.45250192284584045, 0.7898479104042053], 'sparse': [0.0, 0.18017578125], 'dense': [0.349853515625, 0.67822265625], 'sparse+dense': [0.2332356721162796, 0.51220703125], 'colbert+sparse+dense': [0.3209421634674072, 0.6232633590698242]}


In [12]:
similarity_scores['colbert']

[0.45250192284584045, 0.7898479104042053]

In [ ]:
from smve_lab.scifact import load_scifact

corpus, queries, qrels_ds, doc_ids, doc_texts, query_ids, query_texts = load_scifact()
print(f"{len(doc_ids)} docs, {len(query_ids)} queries")

In [ ]:
from smve_lab.bge_m3 import encode_texts
from smve_lab.config import ARTIFACTS_DIR
from smve_lab.storage import get_colbert, get_sparse, load_combined, load_separate, save_embeddings

save_dir = ARTIFACTS_DIR / "embeddings" / "scifact_bgem3"  # gitignored; large binary output

In [ ]:
doc_emb = encode_texts(model, doc_texts, cfg)
save_embeddings(doc_emb, doc_ids, save_dir, "docs")
print("colbert size (GB):", doc_emb["colbert_flat"].nbytes / 1e9)

In [ ]:
query_emb = encode_texts(model, query_texts, cfg)
save_embeddings(query_emb, query_ids, save_dir, "queries")

In [ ]:
# Round-trip check: does what we saved come back identical via both load paths?
emb2, ids2 = load_combined(save_dir, "docs")
emb3, ids3 = load_separate(save_dir, "docs")

N = len(doc_ids)
assert ids2 == ids3 == doc_ids
assert emb2["dense"].shape[0] == N == len(emb2["sparse_indptr"]) - 1 == len(emb2["colbert_offsets"]) - 1
assert np.array_equal(emb2["dense"], emb3["dense"])
assert np.array_equal(get_colbert(emb2, 5), get_colbert(emb3, 5))
assert get_sparse(emb2, 5) == get_sparse(emb3, 5)
print("OK:", emb2["dense"].shape, len(get_sparse(emb2, 0)), get_colbert(emb2, 0).shape)